# Lecture 4.1 demo: functions, objects, models

ASM 532, Lecture 4.1. The code from the slides, in the same order, as cells to run top to bottom:

1. **Function design:** functions with inputs, outputs and a docstring you can test
2. **Objects in Python:** the `farm.py` read-along: classes, inheritance, encapsulation and polymorphism
3. **From classes to models:** `Field` and `Observation` as plain Python classes, the step before they become Django models

Lines marked *farm.py* are the 2024 script's own notes.

# 1. Function design

## Functional Programming

In [ ]:
def greet():
  print ("Hello World!")

greet()

In [ ]:
def greet(name):
  print ("Hello " + name)

greet("Bessy")

## Function with variable number of arguments (*args)

`*args` is a variable-length tuple of arguments.

In [ ]:
def greet(*args):
    for name in args:
      print(f"Hello {name}")

greet("Bessy", "Sunny", "Clover")

## Function can have outputs, aka a "return" value

In [ ]:
def ac_to_sqft(value):
  answer = value*43560
  return answer

sqft = ac_to_sqft(15)
print(sqft)

## Function Design, in short

Write the function header, a little description, a "type contract" (what each argument is going to be), examples of expected behavior, then the function itself. Test it: the examples in the docstring are tests, and `doctest` runs them and checks the answers.

In [ ]:
def get_area(width, height):
  """Get the area of a rectangle.
  @type width, height: int
  >>> get_area(2, 5)
  10
  >>> get_area(100,42)
  4200
  """

  area = width*height

  return(area)

In [ ]:
import doctest
doctest.run_docstring_examples(get_area, globals(), verbose=True)

# 2. Objects in Python

The `farm.py` read-along, one cell per slide, with the slides' notes.

## Python `class`

Used to represent objects. Indentation matters. Note the use of the colon.

Class = `Animal`. Object = this specific animal instance.

This is a bad class! Not reusable! All hardcoded!

- `animal_type`: make each kind of animal its own class.
- `name`: pass the name as an argument when the object is created.
- `age`: store birthdate, calculate age.
- `breed`: breeds vary by animal.

In [ ]:
class Animal:
    animal_type = "cow"
    name = "Bessy"
    age = 10
    breed = "Holstein"

print(Animal.name, Animal.age)

## An abstract, reusable `class Animal`

- `__init__()` is a "constructor" method used to create an object.
- See function documentation: `help(function_name)`.
- `self` is self-referential (this object).
- Order within the function doesn't matter; order when passing into the function matters, hence documentation matters.

In [ ]:
from datetime import date

class Animal:
    def __init__(self, name, birth_year, birth_month, birth_day, weight):
        self.name = name
        self.birthdate = date(birth_year, birth_month, birth_day)
        self.weight = weight

## Instantiate "Bessy"

- `Animal("Bessy", 2010, 4, 1, 200)` calls the class like a function. The object is constructed using `__init__()`. Values must be in the same order as its arguments. This also is why function headers are important - they tell the user what to expect.
- `print(f"hello {bessy.name}")`: the `f` makes a "formatted string". Whatever is inside `{ }` is run, so `{bessy.name}` = `Bessy`. Output printed: `hello Bessy`

In [ ]:
bessy = Animal("Bessy", 2010, 4, 1, 200)
print(f"hello {bessy.name}")

## Inheritance: Dodo is empty class stub

- The brackets denote the parent being inherited: `class Dodo(Animal)` = `Dodo` is a child of `Animal`.
- `Dodo` inherits everything an `Animal` has: attributes and methods.
- `pass` means "nothing new to add yet".

In [ ]:
class Dodo(Animal):
    pass

## Inheritance: Goat is a child class

- Note: in code, `breed` arrives through the `Goat` and `Cow` subclasses; in the Farm tracker data model, `breed` sits on the one Animal table, next to a species column.
- Classes tell Python what kind of animal it is - a table has no classes, so it needs a column. Kleppmann calls this the object-relational mismatch.
- In the line `goat_secret_name = "secret goaty goat"`, the lack of `self.` means that this is a local variable - not visible outside the `__init__` constructor method. Only `self.goat_secret_name` is kept.

In [ ]:
class Goat(Animal):
    def __init__(self, name, birth_year, birth_month, birth_day, weight, breed):
        super().__init__(name, birth_year, birth_month, birth_day, weight)

        self.breed = breed
        goat_secret_name = "secret goaty goat"
        self.goat_secret_name = "secret goaty goat"

## At last, my encapsulated operations!

Hide the birthdate, calculate the age.

- `(today.month, today.day) < (...)` is `True` or `False`. In Python, `True` = 1: it takes off a year if this year's birthday hasn't happened yet.
- `details()` calls `self.calculate_age()`: a method can use the object's other methods through `self`.

This cell re-defines `Animal`, now with two methods.

In [ ]:
class Animal:
    def __init__(self, name, birth_year, birth_month, birth_day, weight):
        self.name = name
        self.birthdate = date(birth_year, birth_month, birth_day)
        self.weight = weight

    def calculate_age(self):
        today = date.today()
        age = today.year - self.birthdate.year - ((today.month, today.day) < (self.birthdate.month, self.birthdate.day))
        return age

    def details(self):
        print(f"{self.name} is {self.calculate_age()} years old today!")

Notebooks are sequential, so run `Goat` again to pick up these methods. `Goat` was built from the *old* `Animal`, which had no methods, so it doesn't have them yet. The next cell is the same `Goat` code as above.

In [ ]:
class Goat(Animal):
    def __init__(self, name, birth_year, birth_month, birth_day, weight, breed):
        super().__init__(name, birth_year, birth_month, birth_day, weight)

        self.breed = breed
        goat_secret_name = "secret goaty goat"
        self.goat_secret_name = "secret goaty goat"

## Goat tried to keep secrets

- Goat inherited `details()` and `calculate_age()` from its parent class: `Animal`.
- We can access the goat's secret name. Why is that bad? Any code can read/change the goat.
- Python has no truly private attributes. The convention is a leading underscore (`self._secret_name`) and associated method.

In [ ]:
sunny = Goat("Sunny", 2009, 4, 1, 20, "highland")
sunny.details()
print(sunny.goat_secret_name)

## Cow is an example of polymorphism

Overriding the previously defined method.

In [ ]:
class Cow(Animal):
    def __init__(self, name, birth_year, birth_month, birth_day, weight, breed):
        super().__init__(name, birth_year, birth_month, birth_day, weight)
        self.breed = breed

    def details(self):
        print(f"{self.name} is {self.calculate_age()} years old today. She is a {self.breed} weighing {self.weight} pounds.")

In [ ]:
bessy = Cow("Bessy", 2010, 4, 1, 200, "Holstein")
bessy.details()

# 3. From classes to models

`farmnotes` is a small app for a farmer to make quick notes about what they see while walking their fields. It has two objects:

- **Field**: the Farm tracker's Field from Lecture 3.3: name, acres, field_type (pasture, cropland), note
- **Observation**: a note about one field: title, author, content, type (from a lookup list), date

First as plain Python classes. In the lecture, the same two classes move into `farmnotes/models.py`.

In [ ]:
from datetime import date

FIELD_TYPES = ["pasture", "cropland"]
OBSERVATION_TYPES = ["weather", "crop", "soil", "water", "pest", "other"]

class Field:
    def __init__(self, name, acres, field_type, note=""):
        self.name = name
        self.acres = acres
        self.field_type = field_type
        self.note = note

    def __str__(self):
        return f"{self.name} ({self.field_type}, {self.acres} acres)"

class Observation:
    def __init__(self, field, observation_title, author, observation_content, observation_type, observation_date):
        self.field = field
        self.observation_title = observation_title
        self.author = author
        self.observation_content = observation_content
        self.observation_type = observation_type
        self.observation_date = observation_date

    def __str__(self):
        return f"{self.observation_title} by {self.author} on {self.observation_date}"

In [ ]:
north = Field("North pasture", 12.5, "pasture")
fence = Observation(north, "Fence down", "Sam", "East fence post broken near the gate.", "other", date(2026, 10, 6))

print(north)
print(fence)
print(fence.field)

## The same classes as Django models

In `farmnotes/models.py`, each class inherits from `models.Model`, and each attribute becomes a model field with a type. Django then:

- builds the database tables (`makemigrations`, `migrate`)
- turns `observation.field` into a **foreign key**, so `north.observation_set.all()` finds a field's observations
- offers the `choices=` lookup lists as drop-downs in the admin

The full file is in `demos/myapp/farmnotes/models.py`. It doesn't run in this notebook, because Django models need a Django project around them.

```python
from django.db import models

class Field(models.Model):
    name = models.CharField(max_length=200)
    acres = models.FloatField()
    field_type = models.CharField(choices=FIELD_TYPES, max_length=20)
    note = models.CharField(max_length=1000, blank=True)

class Observation(models.Model):
    field = models.ForeignKey(Field, on_delete=models.CASCADE)
    observation_title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    observation_content = models.CharField(max_length=1000)
    observation_type = models.CharField(choices=OBSERVATION_TYPES, max_length=100)
    observation_date = models.DateField('date observed')
```